# In-context fine-tune (multi-pair) + eval
Teach M_pop the GENERAL skill: read a person's task-A transcript, predict their task-B behavior. Fine-tune on MANY within-domain [A+B] pairs at once, then evaluate real-A vs floor vs shuffled-A on a held-out pair.

**GPU:** A100 -> `MAXLEN = 6144`. L4 -> `MAXLEN = 4096` (avoids OOM).

In [ ]:
from google.colab import drive; drive.mount('/content/drive')
import os
if not os.path.exists('/content/sro-minitaur-transfer'):
    !git clone https://github.com/YifeiCAO/sro-minitaur-transfer.git
!cd /content/sro-minitaur-transfer && git pull
%cd /content/sro-minitaur-transfer
!pip -q install -e . && pip -q install -r requirements-model.txt
from huggingface_hub import login; login()  # paste hf_ token

In [ ]:
MAXLEN = 6144   # A100. Change to 4096 if on L4.
print('max_seq_len =', MAXLEN)

### 1) Fine-tune on many within-domain [A+B] pairs -> mpop_ic on Drive (~2-3h)
Trains one model over all within-domain source->target pairs in the starting subset (capped at 4000 examples).

In [ ]:
!cd /content/sro-minitaur-transfer && python scripts/finetune_incontext.py \
    --mpop /content/drive/MyDrive/sro_minitaur/mpop \
    --subset starting_subset --pairs within \
    --out /content/drive/MyDrive/sro_minitaur/mpop_ic \
    --max-seq-len {MAXLEN} --max-examples 4000 --epochs 1

### 2) Evaluate on a held-out pair (directed_forgetting -> recent_probes)
`mean_real_minus_floor` < 0 => A context helps. `mean_real_minus_shuffled` < 0 (p<.05) => person-specific transfer.

In [ ]:
!cd /content/sro-minitaur-transfer && python scripts/run_incontext.py \
    --mpop /content/drive/MyDrive/sro_minitaur/mpop_ic \
    --source directed_forgetting --target recent_probes --max-seq-len {MAXLEN}

### 3) (optional) Try another within-domain pair with the SAME model

In [ ]:
!cd /content/sro-minitaur-transfer && python scripts/run_incontext.py \
    --mpop /content/drive/MyDrive/sro_minitaur/mpop_ic \
    --source kirby --target discount_titrate --max-seq-len {MAXLEN}

### Read a result

In [ ]:
import json
print(json.load(open('/content/drive/MyDrive/sro_minitaur/results/incontext/directed_forgetting_recent_probes.json')))